# MLflow na prática

Você treinou vinte variações esta semana. Qual era a boa? Com que
hiperparâmetros? Deu quanto no teste? O arquivo está em qual pasta?

Este notebook roda inteiro no Colab e faz o ciclo completo: treinar,
registrar, comparar, versionar, promover e carregar pelo apelido.

A única coisa que não dá para fazer aqui é abrir a **interface** do MLflow —
e é por isso que a comparação abaixo é feita com `search_runs()`, que é
exatamente a consulta que a tela executa.

Nada aqui depende de outro notebook ou de outra pasta.

In [ ]:
try:
    import mlflow
except ImportError:
    !pip install -q mlflow
    import mlflow

print('mlflow', mlflow.__version__)

## 1. Onde o registro fica salvo

Uma linha. Um arquivo SQLite ao lado do notebook — sem servidor, sem conta.

> **Por que SQLite e não a pasta `mlruns` padrão?** Porque o Model Registry
> (seções 5 e 6) **não funciona** com arquivos soltos. Com banco, funciona.

In [ ]:
import mlflow, mlflow.sklearn
from sklearn.datasets import load_wine
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import ConfusionMatrixDisplay, f1_score
import matplotlib.pyplot as plt

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('wine-arvore')

X, y = load_wine(return_X_y=True, as_frame=True)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
print(f'{len(X)} vinhos, {X.shape[1]} medidas, {y.nunique()} castas')
print(f'treino {len(X_treino)} | teste {len(X_teste)}  (estratificado)')

## 2. `autolog()` — uma linha, e o registro acontece

A partir daqui, todo `fit()` salva sozinho os hiperparâmetros, as métricas de
treino, o modelo serializado, o formato da entrada, o usuário e o horário.

O que ele **não** salva é o que ele não conhece: a validação cruzada acontece
antes do `fit`, e a métrica que interessa a você tem o nome que você der.
Essas quatro linhas são o *tracking manual*.

In [ ]:
mlflow.sklearn.autolog()

VARIACOES = [
    {'max_depth': 2, 'criterion': 'gini'},
    {'max_depth': 3, 'criterion': 'gini'},
    {'max_depth': 5, 'criterion': 'gini'},
    {'max_depth': 3, 'criterion': 'entropy'},
    {'max_depth': None, 'criterion': 'entropy', 'min_samples_leaf': 5},
]

for variacao in VARIACOES:
    nome = '-'.join(f'{c}={v}' for c, v in variacao.items())
    with mlflow.start_run(run_name=nome):
        modelo = DecisionTreeClassifier(random_state=42, **variacao)
        notas = cross_val_score(modelo, X_treino, y_treino, cv=5, scoring='f1_macro')
        modelo.fit(X_treino, y_treino)
        predicoes = modelo.predict(X_teste)

        mlflow.log_metric('f1_cv_media', notas.mean())
        mlflow.log_metric('f1_cv_desvio', notas.std())
        mlflow.log_metric('f1_teste', f1_score(y_teste, predicoes, average='macro'))

        ConfusionMatrixDisplay.from_predictions(y_teste, predicoes)
        plt.title(nome); plt.savefig('matriz.png', dpi=80, bbox_inches='tight'); plt.close()
        mlflow.log_artifact('matriz.png')

        print(f'{nome:45} f1_cv={notas.mean():.4f} (+-{notas.std():.4f})')

## 3. Comparar os runs

Na interface, isto é a tela **Experiments**: uma linha por run, colunas
filtráveis (`metrics.f1_cv_media > 0.89`), e o botão **Compare** para ver dois
lado a lado. A consulta por trás é esta:

In [ ]:
runs = mlflow.search_runs(experiment_names=['wine-arvore'],
                         order_by=['metrics.f1_cv_media DESC'])
runs[['tags.mlflow.runName', 'metrics.f1_cv_media', 'metrics.f1_cv_desvio', 'metrics.f1_teste']]

**Olhe as duas primeiras linhas.** No conjunto de teste elas costumam empatar
no mesmo valor — pelo teste, não dá para escolher. Quem desempata é a
validação cruzada, que olhou cinco divisões em vez de uma.

Sem registro, esse empate passa despercebido e você escolhe pelo número que
estava na tela.

## 4. MLflow Models: `flavor` e assinatura

O run guarda o que você registrou. O **modelo** é guardado à parte, e não
como arquivo solto: como uma pasta, com o `MLmodel` de documento de identidade.

In [ ]:
import os

melhor = runs.iloc[0]

print('O que ficou guardado no run:')
raiz = mlflow.artifacts.download_artifacts(run_id=melhor['run_id'])
for arquivo in sorted(os.listdir(raiz)):
    print('  ', arquivo)

print('\nA pasta do modelo, que é um artefato à parte:')
pasta = mlflow.artifacts.download_artifacts(f"runs:/{melhor['run_id']}/model")
for arquivo in sorted(os.listdir(pasta)):
    print('  ', arquivo)

`matriz.png` é a nossa, do `log_artifact`. O `estimator.html` e o
`training_confusion_matrix.png` **ninguém pediu**: o `autolog()` gerou.

E o modelo não é um arquivo, é uma pasta — com o `MLmodel` (os metadados), o
`model.pkl` e as dependências para reconstruir o ambiente.

In [ ]:
caminho = mlflow.artifacts.download_artifacts(f"runs:/{melhor['run_id']}/model/MLmodel")
print(open(caminho).read()[:1200])

Dois **flavors**: `sklearn` (o objeto original, com `.feature_importances_`) e
`python_function`. O `pyfunc` é o genérico: **qualquer** modelo do MLflow —
sklearn, PyTorch, TensorFlow, XGBoost — carrega com a mesma função e responde
ao mesmo `.predict()`. É por isso que quem consome o modelo não precisa saber
em que framework ele foi treinado.

Repare também na **assinatura**: 13 colunas com nome e tipo. O `autolog()`
inferiu isso do `fit`, e é ela que faz a API recusar uma entrada malformada.

## 5. Model Registry: versão, não sobrescrita

Depois de 50 runs, ninguém lembra qual é o oficial. Registrar sob o **mesmo
nome** não sobrescreve nada: cria a versão seguinte, com histórico de quem
registrou, quando, e a partir de qual run.

In [ ]:
from mlflow import MlflowClient

cliente = MlflowClient()
for posicao, apelido in [(0, 'champion'), (1, 'challenger')]:
    run = runs.iloc[posicao]
    versao = mlflow.register_model(f"runs:/{run['run_id']}/model", 'wine-arvore')
    cliente.set_registered_model_alias('wine-arvore', apelido, versao.version)
    print(f"{apelido:11} -> versão {versao.version}  ({run['tags.mlflow.runName']})")

## 6. Aliases: promover é mover um apelido

Versão é número, e número não diz qual está valendo. O **alias** é um apelido
fixo apontando para uma versão — e a aplicação carrega por ele, sem saber run
nem versão.

> Em tutoriais antigos você vai ver *stages* (`Staging`, `Production`,
> `Archived`). **Está obsoleto**: os aliases substituíram esse esquema.

In [ ]:
modelo_campeao = mlflow.pyfunc.load_model('models:/wine-arvore@champion')

predicoes = modelo_campeao.predict(X_teste.head(8))
print('previsto:', [int(p) for p in predicoes])
print('real:    ', list(y_teste.head(8)))

Repare no que essa célula **não** tem: nenhum `run_id`, nenhum caminho de
arquivo, nenhuma menção a `sklearn`. Trocar o modelo campeão em produção vira
mover o apelido — e nada no código de quem consome muda.

## 7. O mesmo modelo como API REST

Isto **não** roda no Colab (o servidor subiria sem ninguém para acessar). Na
sua máquina, com esta pasta:

```bash
mlflow models serve -m "models:/wine-arvore@champion" -p 1234 --env-manager local
```

E, de outro terminal:

```bash
curl -X POST http://localhost:1234/invocations \
  -H 'Content-Type: application/json' -d @exemplo.json
# {"predictions": [0, 1, 0]}
```

Nenhuma linha de código de servidor foi escrita. O mesmo modelo vira imagem de
contêiner com `mlflow models build-docker`.

## 8. Sua vez

Sem copiar o laço da seção 2:

1. Treine **suas** variações de árvore — mexa em `max_depth`, `criterion`,
   `min_samples_leaf`.
2. Registre um **artefato seu** (um gráfico, um CSV de erros).
3. Promova a melhor a `champion` e carregue pelo alias.
4. Troque `DecisionTreeClassifier` por `RandomForestClassifier` e responda:
   **o que muda na célula que carrega o modelo?**

(Nada. Esse é o ponto do `pyfunc`.)

## 9. O que o MLflow não resolve

- **Não avalia por você.** Métrica errada, registrada com rigor, continua errada.
- **Não versiona o dado.** O CSV que mudou ontem continua sendo problema seu.
- **Não reproduz o ambiente sozinho.** Ele salva um `conda.yaml`; quem garante
  a mesma libc e a mesma GPU é contêiner.
- **Não é monitoramento.** Ele registra treino, não o que acontece depois com
  as predições em produção.